# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [26]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 3.55s


In [28]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 0.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [20]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [21]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.35s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [10]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures
import functools

CAT_API_URL = "https://catfact.ninja/fact"


def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper


def download_fact(index=None):
    return requests.get(CAT_API_URL).json().get('fact')


@timer
def run_sequential():
    return [download_fact() for _ in range(20)]


@timer
def run_threads():
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        return list(executor.map(download_fact, range(20)))


print(run_sequential())
print(run_threads())

Czas wykonania run_sequential: 6.7704 s
['On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.', 'Ailurophile is the word cat lovers are officially called.', 'The cat who holds the record for the longest non-fatal fall is Andy. He fell from the 16th floor of an apartment building (about 200 ft/.06 km) and survived.', 'A cat lover is called an Ailurophilia (Greek: cat+lover).', 'Cats purr at the same frequency as an idling diesel engine, about 26 cycles per second.', 'The first cat in space was a French cat named Felicette (a.k.a. “Astrocat”) In 1963, France blasted the cat into outer space. Electrodes implanted in her brains sent neurological signals back to Earth. She survived the trip.', 'Phoenician cargo ships are thought to have brought the first domesticated cats to Europe in about 900 BC.', 'Like humans, cats tend to favor one paw over another', 'The first cat show was organized in 1871 in London. Cat sh

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [23]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

NUM = 20
QUEUE = queue.Queue()


def producer():
    for i in range(1, NUM + 1):
        QUEUE.put(i)
        time.sleep(0.5)
    QUEUE.put(None)


def consumer_odd():
    while True:
        item = QUEUE.get()
        if item is None:
            QUEUE.put(None)
            break
        if item % 2 != 1:
            QUEUE.put(item)
            continue
        print(f'Odd: {item}')


def consumer_even():
    while True:
        item = QUEUE.get()
        if item is None:
            QUEUE.put(None)
            break
        if item % 2 != 0:
            QUEUE.put(item)
            continue
        print(f'Even: {item}')


thread_functions = [producer, consumer_odd, consumer_even]
threads = [threading.Thread(target=func) for func in thread_functions]
[thread.start() for thread in threads]
[thread.join() for thread in threads]

print('Done')

Odd: 1
Even: 2
Odd: 3
Even: 4
Odd: 5
Even: 6
Odd: 7
Even: 8
Odd: 9
Even: 10
Odd: 11
Even: 12
Odd: 13
Even: 14
Odd: 15
Even: 16
Odd: 17
Even: 18
Odd: 19
Even: 20
Done


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [24]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum


if __name__ == "__main__":
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    numbers = [(i,) for i in range(1, 10_001)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(calculate_power_sum, numbers)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")
    print(results[:100])

Praca na 16 procesach (rdzeniach)...
Zakończono w czasie 0.25s.
[100, 2535301200456458802993406410750, 773066281098016996554691694648431909053161283000, 2142584059011987034055949456454883470029603991710390447068500, 9860761315262647567646607066034827870915080438862787559628486633300780, 783982348200085087316028320589669384644572452567545845851686359643396569772850, 3773555927895550989902089063950252946000070398722062967756211219956973369576416070000, 2328041115810841241449652215325003612630249592761070000727017656405007199729527664209597000, 298815737485359115506128987290252080182887634235068807971396831956479052263964955868682786424500, 11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110, 151586735738044972025301708892986004538907643200423130764611253073704837305075384446341275447204379090600, 903468812967431457364455530135311088744861249295157565960947488112641730178314061495749399960208748420625500, 2685946370213136911303662291796103